# 3. Analysis of human data

Descriptive analyses of the human trajectories in Experiments 1 and 2: walking speed at obstacle onset,
mean trajectories by condition (Figures 6, 14), proportion and magnitude of speed vs heading control
(Figures 8, 9, 16, 17), passing order by condition (Figures 7a, 15a), and signed minimum predicted distance
(SMPD; Figure 11, Tables 3, 6).

The cells marked *to_csv* write the trial-level tables used by the R scripts in `stats/` to `results/`.
Run `01_import_data.ipynb` first.

In [ ]:
import os
import sys
import json
import random
from itertools import product
from collections import Counter, defaultdict
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.stats import ttest_ind
import numpy as np
from numpy.linalg import norm
from numpy import sqrt
from sklearn.metrics import accuracy_score
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
from math import pi
import pickle
import matplotlib.pyplot as plt
from matplotlib import rcParams, animation
from packages import data_container
from packages.data_container import Data
from packages.helper import play_trajs, rotate, sp2a, v2sp, psi, beta, d_theta, d_psi, sp2v, dist, min_dist, \
    vector_angle, signed_angle, side, inner, theta, min_sep, traj_speed, play_trajs_variable
from packages.ode_simulator import ODESimulator
from parameters import approaches, avoidances
from config import processed_path, RESULTS_DIR
# For pickle to load the Data object, which is defined in packages.data_container
sys.modules['data_container'] = data_container
data = None
def load_data(filename):
    global data
    with open(processed_path(filename), 'rb') as f:
        data = pickle.load(f)

os.makedirs(RESULTS_DIR, exist_ok=True)

## Experiment 1 (`Cohen_movObst1`)

In [ ]:
load_data('Cohen_movObst1_data')

In [ ]:
'''mean and sd of participant speed when obstacle appeared'''
speeds = []
obst_onset = data.info['obst_onset']
velocities = data.info['v_subj']
import pandas as pd

df_speeds = []
for i in range(len(obst_onset)):
    if i not in data.dump:
        t0 = obst_onset[i]
        v = velocities[i][t0]
        subj_id = data.info['subj_id'][i]
        trial_id = data.info['trial_id'][i]
        speed = norm(v)
        df_speeds.append({'subj_id': subj_id, 'trial_id': trial_id, 'subj_speed_at_obst_onset': speed})
df_speeds = pd.DataFrame(df_speeds)
mean_speed = df_speeds['subj_speed_at_obst_onset'].mean()
sd_speed = df_speeds['subj_speed_at_obst_onset'].std()
print("Mean subj_speed_at_obst_onset:", mean_speed)
print("SD subj_speed_at_obst_onset:", sd_speed)

# df_speeds.to_csv('subj_speed_at_obst_onset.csv', index=False)

In [ ]:
'''effective data length'''
%matplotlib qt
l = []
sim = ODESimulator(data=data, ref=[0,1])
for i in range(len(data.trajs)):
    if i not in data.dump and data.info['obst_speed'][i] > 0 and abs(data.info['obst_angle'][i]) != 180:
        t0 = data.info['obst_onset'][i]
        t1 = data.info['obst_out'][i]
        if t0 and t1:
            l.append((t1 - t0) / data.Hz)
plt.hist(l, bins=40)
# plt.plot(l)

In [ ]:
'''Animate data'''
load_data('Cohen_movObst1_data')
############
subject = 3
trial = 111
############
%matplotlib qt
i = subject * 160 + trial - 1
i = 615
# p_obst = np.array(data.info['p_obst'][i])
t0 = data.info['obst_onset'][i]
t1 = data.info['obst_out'][i]
p_goal = data.info['p_goal'][i]
p_subj = data.info['p_subj'][i]
p_obst = data.info['p_obst'][i]
trajs = [p_goal, p_obst, p_subj]
ws = [data.info['w_goal'], data.info['w_obst'], 0.4]
title = 'subj ' + str(data.info['subj_id'][i]) + ' trial ' + str(data.info['trial_id'][i]) + ' obst_angle: ' + str(data.info['obst_angle'][i]) + ' obst_speed: ' + str(data.info['obst_speed'][i])
play_trajs(trajs, ws, data.Hz, title=title, save=False)


In [ ]:
'''Time averaged traj by condition'''
'''
In within subject design, the standard error is within subject sum of squares (s_w) divided by root(2n(m-1))
n is the number of subjects m is number of conditions.
Confidence interval is +-z*SE, for 95% CI z = 1.96
'''
%matplotlib qt
for angle in set(np.abs(data.info['obst_angle'])):
    for speed in set(data.info['obst_speed']):
        fig = plt.figure()
        ax = fig.add_subplot(1,1,1)
        ax.set_title(f'obst_angle {angle}, obst_speed {speed}')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_aspect('equal')
        ax.set_xlim(-1.5, 1.5)
        ax.set_ylim(-0.5, 8.5)
        front = []
        behind = []
        len_front = float('inf')
        len_behind = float('inf')
        # Plot trials
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            traj = np.array(data.info['p_subj'][i])
            order = data.info['pass_order'][i]
            if data.info['obst_angle'][i] < 0:
                traj[:, 0] *= -1
                
            ax.plot(traj[:, 0], traj[:, 1], color='k', linewidth=0.1, alpha=0.5)
            m = len(traj) // 2
            if order == 1:
                front.append(traj)
                len_front = min(len(traj), len_front)
            elif order == -1:
                behind.append(traj)
                len_behind = min(len(traj), len_behind)
        # Plot average trials
        front = [traj[:len_front] for traj in front]
        behind = [traj[:len_behind] for traj in behind]
        front = np.mean(front, axis=0)
        behind = np.mean(behind, axis=0)
        ax.plot(front[:, 0], front[:, 1], color='k')
        ax.plot(behind[:, 0], behind[:, 1], color='k')

In [ ]:
traj = np.array(data.info['p_obst'][0])
np.array(data.info['obst_angle'][0])

In [ ]:
'''Space averaged traj by condition'''
%matplotlib qt
load_data('Cohen_movObst1_data')
y_max = float('inf')
y_min = -float('inf')
for i in range(len(data.trajs)):
    if i in data.dump:
        continue
    traj = np.array(data.info['p_subj'][i])
    y_max = min(y_max, max(traj[:, 1]))
    y_min = max(y_min, min(traj[:, 1]))
print(f'common y range among all trials is [{y_min}, {y_max}]')

# Compute y positions
y_step = 0.01
y = np.linspace(y_min, y_max, int((y_max - y_min) / y_step))

angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
speeds = sorted(set([abs(x) for x in data.info['obst_speed'] if x]))
fig, axs = plt.subplots(3, 3, figsize=(6.5, 12), constrained_layout=True)
for ix, angle in enumerate(angles):
    for iy, speed in enumerate(speeds):
        # axs[ix, iy].set_title(f'{angle}°, {speed} m/s')
        axs[ix, iy].set_xlabel('x (m)')
        axs[ix, iy].set_ylabel('y (m)')
        axs[ix, iy].set_aspect('equal')
        axs[ix, iy].set_xlim(-1.5, 1.5)
        axs[ix, iy].set_ylim(-0.2, 8)
        front = []
        behind = []
        # Plot trials
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            traj = np.array(data.info['p_subj'][i])
            # Flip the traj if the obstacle is on the left side
            if data.info['obst_angle'][i] > 0:
                traj[:,0] = -traj[:,0]
            x = np.interp(y, traj[:,1], traj[:,0])
            order = data.info['pass_order'][i]
            if order == 1:
                front.append(x)
                axs[ix, iy].plot(traj[:, 0], traj[:, 1], color='r', linewidth=0.1, alpha=0.5)
            elif order == -1:
                behind.append(x)
                axs[ix, iy].plot(traj[:, 0], traj[:, 1], color='b', linewidth=0.1, alpha=0.5)
        # Plot average trials
        front = np.mean(front, axis=0)
        behind = np.mean(behind, axis=0)
        axs[ix, iy].plot(front, y, color='r')
        axs[ix, iy].plot(behind, y, color='b')

In [ ]:
'''Temporally averaged speed'''
%matplotlib qt
load_data('Cohen_movObst1_data')
        
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
speeds = sorted(set([abs(x) for x in data.info['obst_speed'] if x]))
fig, axs = plt.subplots(3, 3, figsize=(10, 7), constrained_layout=True)
for ix, angle in enumerate(angles):
    for iy, speed in enumerate(speeds):
        axs[ix, iy].set_title(f'{angle}°, {speed} m/s')
        axs[ix, iy].set_xlabel('Normalized time (%)')
        axs[ix, iy].set_ylabel('Speed (m/s)')
        axs[ix, iy].set_ylim(0, 2)
        front = np.zeros(100)
        nfront = 0
        behind = np.zeros(100)
        nbehind = 0
        # Plot trials
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            v0 = data.info['v_subj'][i]
            ref = [9, 11] if i % 2 == 0 else [-9, -11]
            s, h = v2sp(v0, ref=ref)
            t = np.linspace(0, 100, len(s))
            order = data.info['pass_order'][i]
            if order == 1:
                front += interp1d(t, s)(range(100))
                nfront += 1
                axs[ix, iy].plot(t, s, color='r', linewidth=0.1, alpha=0.5)
            elif order == -1:
                behind += interp1d(t, s)(range(100))
                nbehind += 1
                axs[ix, iy].plot(t, s, color='b', linewidth=0.1, alpha=0.5)
        # Plot average trials
        front = front / nfront
        behind = behind / nbehind
        axs[ix, iy].plot(range(100), front, color='r')
        axs[ix, iy].plot(range(100), behind, color='b')

In [ ]:
'''Temporally averaged heading'''
%matplotlib qt
load_data('Cohen_movObst1_data')
        
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
speeds = sorted(set([abs(x) for x in data.info['obst_speed'] if x]))
fig, axs = plt.subplots(3, 3, figsize=(10, 7), constrained_layout=True)
for ix, angle in enumerate(angles):
    for iy, speed in enumerate(speeds):
        axs[ix, iy].set_title(f'{angle}°, {speed} m/s')
        axs[ix, iy].set_xlabel('Normalized time (%)')
        axs[ix, iy].set_ylabel('Heading (°)')
        axs[ix, iy].set_ylim(-50, 50)
        front = np.zeros(100)
        nfront = 0
        behind = np.zeros(100)
        nbehind = 0
        # Plot trials
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            v0 = data.info['v_subj'][i]
            if data.info['obst_angle'][i] < 0:
                v0[:,0] = -v0[:,0]
            s, h = v2sp(v0)
            h = h * 180 / np.pi
            t = np.linspace(0, 100, len(h))
            order = data.info['pass_order'][i]
            if order == 1:
                front += interp1d(t, h)(range(100))
                nfront += 1
                axs[ix, iy].plot(t, h, color='r', linewidth=0.1, alpha=0.5)
            elif order == -1:
                behind += interp1d(t, h)(range(100))
                nbehind += 1
                axs[ix, iy].plot(t, h, color='b', linewidth=0.1, alpha=0.5)
        # Plot average trials
        front = front / nfront
        behind = behind / nbehind
        axs[ix, iy].plot(range(100), front, color='r')
        axs[ix, iy].plot(range(100), behind, color='b')

### Cohen_movObst1: Speed vs Heading control

In [ ]:
'''Speed and heading control in each trial'''
load_data('Cohen_movObst1_data')

rows = []
for i in range(len(data.trajs)):
    if i not in data.dump:
        t0 = data.info['obst_onset'][i]
        t1 = data.info['obst_out'][i]
        v = data.info['v_subj'][i][t0:t1]
        a = data.info['a_subj'][i][t0:t1]
        speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
        heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
        obst_angle = data.info['obst_angle'][i]
        obst_speed = data.info['obst_speed'][i]
        subj_id = data.info['subj_id'][i]
        trial_id = data.info['trial_id'][i]
        rows.append({'subj_id': subj_id, 'trial_id': trial_id, 'obst_angle': obst_angle, 'obst_speed': obst_speed, 'speed_ctrl': np.mean(speed_ctrl), 'heading_ctrl': np.mean(heading_ctrl)})
df = pd.DataFrame(rows)

df.to_csv(os.path.join(RESULTS_DIR, 'Cohen_movObst1_speed_heading_ctrl.csv'), index=False)

In [ ]:
'''Speed control vs heading control all trials'''
load_data('Cohen_movObst1_data')
%matplotlib qt
sc = []
hc = []
for i in range(len(data.trajs)):
    if i in data.dump: continue
    t0 = data.info['obst_onset'][i]
    t1 = data.info['obst_out'][i]
    v = data.info['v_subj'][i][t0:t1]
    a = data.info['a_subj'][i][t0:t1]
    speed_ctrl = np.array([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
    heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
    sc.extend(speed_ctrl)
    hc.extend(heading_ctrl)
    plt.plot(speed_ctrl, color='r', linewidth=0.1, alpha=0.5)
    plt.plot(heading_ctrl, color='b', linewidth=0.1, alpha=0.5)
plt.title(f'mean speed control {np.mean(sc):.2f} mean heading control {np.mean(hc):.2f} ratio {np.mean(sc) / np.mean(hc): .2f}')

In [ ]:
'''Signed speed vs heading control by conditions (abs obst_angle)'''
%matplotlib qt
for angle in set(np.abs(data.info['obst_angle'])):
    for speed in set(data.info['obst_speed']):
        fig = plt.figure()
        ax = fig.add_subplot(1,1,1)
        ax.set_title(f'obst_angle {angle}, obst_speed {speed}')
        ax.set_ylabel('acceleration (m/s**2)')
        ax.set_xlabel('time (s)')
        ax.set_xlim(0, 4.5)
        ax.set_ylim(-1.5, 1.5)
        sc = []
        hc = []
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = np.array([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
            heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
            # Check the side of heading ctrl: front (+) or behind (-) driving
            pass_side = np.sign(inner(rotate(a, pi/2), v)) * np.sign(data.info['obst_angle'][i]) * -1
            heading_ctrl *= pass_side
            sc.extend(speed_ctrl)
            hc.extend(heading_ctrl)
            plt.plot(np.linspace(0, len(speed_ctrl)/data.Hz, len(speed_ctrl)), speed_ctrl, color='r', linewidth=0.1, alpha=0.5)
            plt.plot(np.linspace(0, len(speed_ctrl)/data.Hz, len(speed_ctrl)), heading_ctrl, color='b', linewidth=0.1, alpha=0.5)
        ax.set_title(f'obst_angle {angle}, obst_speed {speed}, \n mean speed control {np.mean(np.abs(sc)):.2f} mean heading control {np.mean(np.abs(hc)):.2f} ratio {np.mean(np.abs(sc)) / np.mean(np.abs(hc)): .2f}')

In [ ]:
'''Signed speed vs heading control by subject'''
for s in set(data.info['subj_id']):
    fig = plt.figure()
    ax = fig.add_subplot(1,1,1)
    ax.set_title(f'subject {s}')
    ax.set_ylabel('acceleration (m/s**2)')
    ax.set_xlabel('time (s)')
    ax.set_xlim(0, 4.5)
    ax.set_ylim(-1.5, 1.5)
    sc = []
    hc = []
    for i in range(len(data.trajs)):
        if s != data.info['subj_id'][i] or i in data.dump:
            continue
        t0 = data.info['obst_onset'][i]
        t1 = data.info['obst_out'][i]
        v = data.info['v_subj'][i][t0:t1]
        a = data.info['a_subj'][i][t0:t1]
        speed_ctrl = np.array([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
        heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
        # Check the side of heading ctrl: front (+) or behind (-) driving
        pass_side = np.sign(inner(rotate(a, pi/2), v)) * np.sign(data.info['obst_angle'][i]) * -1
        heading_ctrl *= pass_side
        sc.extend(speed_ctrl)
        hc.extend(heading_ctrl)
        plt.plot(np.linspace(0, len(speed_ctrl)/data.Hz, len(speed_ctrl)), speed_ctrl, color='r', linewidth=0.1, alpha=0.5)
        plt.plot(np.linspace(0, len(speed_ctrl)/data.Hz, len(speed_ctrl)), heading_ctrl, color='b', linewidth=0.1, alpha=0.5)
    ax.set_title(f'subject {s}, \n mean speed control {np.mean(np.abs(sc)):.2f} mean heading control {np.mean(np.abs(hc)):.2f} ratio {np.mean(np.abs(sc)) / np.mean(np.abs(hc)): .2f}')

In [ ]:
'''Speed/heading control ratio by condition (signed angle)'''
%matplotlib qt
conditions = np.zeros((6, 3)) # speed by angle
angle_ins = {-110: 0, -90: 1, -70: 2, 70: 3, 90: 4, 110: 5}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in set(data.info['obst_angle']):
    for speed in set(data.info['obst_speed']):
        sc = []
        hc = []
        for i in range(len(data.trajs)):
            if angle != data.info['obst_angle'][i] or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
            heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
            sc.extend(speed_ctrl)
            hc.extend(heading_ctrl)
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(sc) / np.mean(hc), 2)

# Heatmap
plt.imshow(conditions, cmap='plasma')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        text = plt.text(j, i, str(conditions[i, j]),
                       ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 6), list(angle_ins.keys()))
plt.xlabel("speed (m/s)")
plt.ylabel("angle (°)")

In [ ]:
'''Speed/heading control ratio by condition (abs angle)'''
%matplotlib qt
load_data('Cohen_movObst1_data')
conditions = np.zeros((3, 3)) # speed by angle
angle_ins = {70: 0, 90: 1, 110: 2}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in set(np.abs(data.info['obst_angle'])):
    for speed in set(data.info['obst_speed']):
        sc = []
        hc = []
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
#             if data.info['pass_order'][i] != -1:
#                 continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
            heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
            sc.extend(speed_ctrl)
            hc.extend(heading_ctrl)
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(sc) / np.mean(hc), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar()
cbar.set_label('Speed/heading control ratio')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.6:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 3), list(angle_ins.keys()))
plt.xlabel("Obstacle speed (m/s)")
plt.ylabel("Obstacle path angle (°)")
# plt.title('Rear passing')

In [ ]:
'''Speed control ratio (by angle)'''
%matplotlib qt
load_data('Cohen_movObst1_data')
conditions = np.zeros((3, 3)) # speed by angle
angle_ins = {70: 0, 90: 1, 110: 2}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in set(np.abs(data.info['obst_angle'])):
    for speed in set(data.info['obst_speed']):
        sc_ratios = []
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
#             if data.info['pass_order'][i] != -1:
#                 continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            av_angle = [np.arccos(np.inner(x, y)/norm(y)/norm(x)) for x, y in zip(a, v)] # a along direction of v
            # Speed control ratio is defined by abs(90 - av_angle)/90
            sc_ratio = [abs(np.pi/2 - x) / (np.pi/2) for x in av_angle]
            sc_ratios.append(np.mean(sc_ratio))
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(sc_ratios), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar()
cbar.set_label('Speed control ratio')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.42:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 3), list(angle_ins.keys()))
plt.xlabel("Obstacle speed (m/s)")
plt.ylabel("Obstacle path angle (°)")
# plt.title('Rear passing')

In [ ]:
'''Signed speed control by condition (abs angle)'''
%matplotlib qt
load_data('Cohen_movObst1_data')
conditions = np.zeros((3, 3)) # speed by angle
angle_ins = {70: 0, 90: 1, 110: 2}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in set(np.abs(data.info['obst_angle'])):
    for speed in set(data.info['obst_speed']):
        sc = []
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = [np.inner(x, y)/norm(y) for x, y in zip(a, v)] # a along direction of v
            sc.extend(speed_ctrl)
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(sc), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar()
cbar.set_label('Speed control ($m/s^2$)')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.08:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 3), list(angle_ins.keys()))
plt.xlabel("Obstacle speed (m/s)")
plt.ylabel("Obstacle path angle (°)")

In [ ]:
'''Heading control by condition (abs angle)'''
%matplotlib qt
conditions = np.zeros((3, 3)) # speed by angle
angle_ins = {70: 0, 90: 1, 110: 2}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in set(np.abs(data.info['obst_angle'])):
    for speed in set(data.info['obst_speed']):
        hc = []
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
            heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
            hc.extend(heading_ctrl)
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(hc), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar()
cbar.set_label('Heading control ($m/s^2$)')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.24:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 3), list(angle_ins.keys()))
plt.xlabel("Obstacle speed (m/s)")
plt.ylabel("Obstacle path angle (°)")

### Cohen_movObst1: Passing order and passing distance

In [ ]:
# Make a pandas df with subj_id, trial_id, obst_angle, obst_speed, pass_order
rows = []

import pandas as pd

for i in range(len(data.trajs)):
    if i not in data.dump:
        subj_id = data.info['subj_id'][i]
        trial_id = data.info['trial_id'][i]
        obst_angle = data.info['obst_angle'][i]
        obst_speed = data.info['obst_speed'][i]
        pass_order = data.info['pass_order'][i]
        rows.append({'subj_id': subj_id, 'trial_id': trial_id, 'obst_angle': obst_angle, 'obst_speed': obst_speed, 'pass_order': pass_order})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(RESULTS_DIR, 'Cohen_movObst1_pass_order.csv'), index=False)

In [ ]:
'''Heatmap and line plot of ahead passing ratio'''
%matplotlib qt
load_data('Cohen_movObst1_data')
name = data.info['experiment_name']
conditions = np.zeros((3, 3)) # angle by speed
angle_ins = {70: 0, 90: 1, 110: 2}
speed_ins = {0.4: 0, 0.6: 1, 0.8: 2}
for angle in angle_ins.keys():
    for speed in speed_ins.keys():
        orders = []
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            orders.append(data.info['pass_order'][i])
        conditions[angle_ins[angle], speed_ins[speed]] = round(np.mean(np.array(orders) > 0), 2)

# Make a pandas df with subj_id, trial_id, obst_angle, obst_speed, pass_order
# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar()
cbar.set_label('Ahead passing ratio')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] >= 0.7:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 3), list(speed_ins.keys()))
plt.yticks(range(0, 3), list(angle_ins.keys()))
plt.xlabel("Obstacle speed (m/s)")
plt.ylabel("Obstacle path angle (°)")

# Line plot
# plt.figure()
# x = list(speed_ins.keys())
# for i, angle in zip(range(conditions.shape[1]), angle_ins.keys()):
#     plt.plot(x, conditions[:, i], label=f'{angle}°', color='r', alpha=(angle-67.5)/90)
# plt.ylabel("Front pass ratio")
# plt.xlabel("speed (m/s)")
# plt.legend()
# plt.title(name)

In [ ]:
'''Minimum Predicted Distance'''
%matplotlib qt
load_data('Cohen_movObst1_data')
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5), constrained_layout=True)
ax.set_title('Data')
ax.set_ylabel('SMPD (m)')
ax.set_xlabel('normalized time (%)')
ax.set_ylim((-3, 3))
last_smpd_front = []
last_smpd_behind = []
for i in range(len(data.trajs)):
    if not data.info['obst_angle'][i] or not data.info['obst_speed'][i] or i in data.dump:
        continue
    t0 = data.info['obst_onset'][i]
    t1 = data.info['obst_out'][i]
    p0 = data.info['p_subj'][i][t0:t1]
    p1 = data.info['p_obst'][i][t0:t1]
    v0 = data.info['v_subj'][i][t0:t1]
    v1 = data.info['v_obst'][i][t0:t1]
    t = np.linspace(0, 100, len(p0))
    smpd = []
    for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
        smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
    ax.plot(t, smpd, 'k', linewidth=0.1, alpha=0.5)
    if smpd[-1] > 0:
        last_smpd_front.append(smpd[-1])
    else:
        last_smpd_behind.append(smpd[-1])
print(np.mean(last_smpd_front), np.std(last_smpd_front), np.mean(last_smpd_behind), np.std(last_smpd_behind))

In [ ]:
'''Minimum Passing Distance by condition'''
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
speeds = sorted(set([abs(x) for x in data.info['obst_speed'] if x]))
fig, axs = plt.subplots(3, 3, figsize=(9, 6), constrained_layout=True)
fig.suptitle('Signed minimum predicted distance (SMPD)')
for ix, angle in enumerate(angles):
    for iy, speed in enumerate(speeds):
        axs[ix, iy].set_title(f'angle {angle}°, speed {speed} m/s')
        axs[ix, iy].set_ylabel('SMPD (m)')
        axs[ix, iy].set_xlabel('normalized time (%)')
        axs[ix, iy].set_ylim((-2, 2))
        for i in range(len(data.trajs)):
            if angle != abs(data.info['obst_angle'][i]) or speed != data.info['obst_speed'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            p0 = data.info['p_subj'][i][t0:t1]
            p1 = data.info['p_obst'][i][t0:t1]
            v0 = data.info['v_subj'][i][t0:t1]
            v1 = data.info['v_obst'][i][t0:t1]
            t = np.linspace(0, 100, len(p0))
            smpd = []
            for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
                smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
            axs[ix, iy].plot(t, smpd, 'k', linewidth=0.1, alpha=0.7)

## Experiment 2 (`Cohen_movObst2`)

In [ ]:
load_data('Cohen_movObst2_data')

In [ ]:
'''mean and sd of participant speed when obstacle appeared'''
speeds = []
obst_onset = data.info['obst_onset']
velocities = data.info['v_subj']
import pandas as pd

df_speeds = []
for i in range(len(obst_onset)):
    if i not in data.dump:
        t0 = obst_onset[i]
        v = velocities[i][t0]
        subj_id = data.info['subj_id'][i]
        trial_id = data.info['trial_id'][i]
        speed = norm(v)
        df_speeds.append({'subj_id': subj_id, 'trial_id': trial_id, 'subj_speed_at_obst_onset': speed})
df_speeds = pd.DataFrame(df_speeds)
mean_speed = df_speeds['subj_speed_at_obst_onset'].mean()
sd_speed = df_speeds['subj_speed_at_obst_onset'].std()
print("Mean subj_speed_at_obst_onset:", mean_speed)
print("SD subj_speed_at_obst_onset:", sd_speed)

df_speeds.to_csv(os.path.join(RESULTS_DIR, 'Cohen_movObst2_subj_speed_at_obst_onset.csv'), index=False)

In [ ]:
'''Plot speed'''
for i in range(len(data.trajs)):
    if data.info['obst_out'][i] and data.info['obst_onset'][i] and i not in data.dump:
        t0, t1 = data.info['obst_onset'][i], data.info['obst_out'][i]
        v_subj = data.info['v_subj'][i]
        s_subj = norm(v_subj, axis=1)
        plt.plot(s_subj[t0:t1], linewidth=0.1, alpha=0.5)

In [ ]:
'''Animate data'''
############
subject = 3
trial = 111
############
%matplotlib qt
i = subject * 160 + trial - 1
i = 79
# p_obst = np.array(data.info['p_obst'][i])
t0 = data.info['obst_onset'][i]
t1 = data.info['obst_out'][i]
p_goal = data.info['p_goal'][i]
p_subj = data.info['p_subj'][i]
p_obst = data.info['p_obst'][i]
trajs = [p_goal, p_obst, p_subj]
ws = [data.info['w_goal'], data.info['w_obst'], 0.4]
title = 'subj ' + str(data.info['subj_id'][i]) + ' trial ' + str(data.info['trial_id'][i]) + ' obst_angle: ' + str(data.info['obst_angle'][i]) + ' obst_dist: ' + str(data.info['obst_dist'][i])
play_trajs(trajs, ws, data.Hz, title=title, save=False)


### Cohen_movObst2: Trajectories

In [ ]:
'''Space averaged traj by condition'''
%matplotlib qt
y_max = float('inf')
y_min = -float('inf')
for i in range(len(data.trajs)):
    if i in data.dump:
        continue
    traj = np.array(data.info['p_subj'][i])
    y_max = min(y_max, max(traj[:, 1]))
    y_min = max(y_min, min(traj[:, 1]))
print(f'common y range among all trials is [{y_min}, {y_max}]')

# Compute y positions
y_step = 0.01
y = np.linspace(y_min, y_max, int((y_max - y_min) / y_step))
# Interpolate x by y
iplot = 1
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
dists = sorted(set([x for x in data.info['obst_dist'] if x]))
fig = plt.figure()
for dist in dists:
    for angle in angles:
        ax = fig.add_subplot(len(dists), len(angles), iplot)
        iplot += 1
        # ax.set_title(f'angle {angle}°, distance {dist} m')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_aspect('equal')
        ax.set_xlim(-2.2, 2.2)
        ax.set_ylim(-0.2, 8.2)
        front = []
        behind = []
        # Plot trials
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i] or angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            traj = np.array(data.info['p_subj'][i])
            # Flip the traj if the obstacle is on the left side
            if data.info['obst_angle'][i] > 0:
                traj[:,0] = -traj[:,0]
            x = np.interp(y, traj[:,1], traj[:,0])
            order = data.info['pass_order'][i]
            if order == 1:
                front.append(x)
                ax.plot(traj[:, 0], traj[:, 1], color='r', linewidth=0.1, alpha=0.5)
            elif order == -1:
                behind.append(x)
                ax.plot(traj[:, 0], traj[:, 1], color='b', linewidth=0.1, alpha=0.5)
        # Plot average trials
        front = np.mean(front, axis=0)
        behind = np.mean(behind, axis=0)
        ax.plot(front, y, color='r')
        ax.plot(behind, y, color='b')

### Cohen_movObst2: Heading vs speed control

In [ ]:
'''Speed and heading control in each trial'''
load_data('Cohen_movObst2_data')

rows = []
for i in range(len(data.trajs)):
    if i not in data.dump:
        t0 = data.info['obst_onset'][i]
        t1 = data.info['obst_out'][i]
        v = data.info['v_subj'][i][t0:t1]
        a = data.info['a_subj'][i][t0:t1]
        speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
        heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
        obst_angle = data.info['obst_angle'][i]
        obst_dist = data.info['obst_dist'][i]
        subj_id = data.info['subj_id'][i]
        trial_id = data.info['trial_id'][i]
        rows.append({'subj_id': subj_id, 'trial_id': trial_id, 'obst_angle': obst_angle, 'obst_dist': obst_dist, 'speed_ctrl': np.mean(speed_ctrl), 'heading_ctrl': np.mean(heading_ctrl)})
df = pd.DataFrame(rows)

df.to_csv(os.path.join(RESULTS_DIR, 'Cohen_movObst2_speed_heading_ctrl.csv'), index=False)

In [ ]:
'''Speed/heading control ratio by condition (abs angle)'''
%matplotlib qt
load_data('Cohen_movObst2_data')
conditions = np.zeros((2, 5)) # speed by angle
angle_ins = {100: 0, 110: 1, 120: 2, 130: 3, 140: 4}
dist_ins = {4.7: 0, 5.8: 1}
for angle in angle_ins.keys():
    for dist in dist_ins.keys():
        sc = []
        hc = []
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
            heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
            sc.extend(speed_ctrl)
            hc.extend(heading_ctrl)
        conditions[dist_ins[dist], angle_ins[angle]] = round(np.mean(sc) / np.mean(hc), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar(orientation="horizontal")
cbar.set_label('Speed/heading control ratio')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.44:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 5), list(angle_ins.keys()))
plt.yticks(range(0, 2), list(dist_ins.keys()))
plt.xlabel("Obstacle path angle (°)")
plt.ylabel("Init obstacle distance (m)")

In [ ]:
'''Speed control ratio by condition (abs angle)'''
%matplotlib qt
load_data('Cohen_movObst2_data')
conditions = np.zeros((2, 5)) # speed by angle
angle_ins = {100: 0, 110: 1, 120: 2, 130: 3, 140: 4}
dist_ins = {4.7: 0, 5.8: 1}
for angle in angle_ins.keys():
    for dist in dist_ins.keys():
        sc_ratios = []
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            av_angle = [np.arccos(np.inner(x, y)/norm(y)/norm(x)) for x, y in zip(a, v)] # a along direction of v
            # Speed control ratio is defined by abs(90 - av_angle)/90
            sc_ratio = [abs(np.pi/2 - x) / (np.pi/2) for x in av_angle]
            sc_ratios.append(np.mean(sc_ratio))
        conditions[dist_ins[dist], angle_ins[angle]] = round(np.mean(sc_ratios), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar(orientation="horizontal")
cbar.set_label('Speed control ratio')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.37:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 5), list(angle_ins.keys()))
plt.yticks(range(0, 2), list(dist_ins.keys()))
plt.xlabel("Obstacle path angle (°)")
plt.ylabel("Init obstacle distance (m)")

In [ ]:
conditions

In [ ]:
'''Speed control by condition (abs angle)'''
%matplotlib qt
load_data('Cohen_movObst2_data')
conditions = np.zeros((2, 5)) # speed by angle
angle_ins = {100: 0, 110: 1, 120: 2, 130: 3, 140: 4}
dist_ins = {4.7: 0, 5.8: 1}
for angle in angle_ins.keys():
    for dist in dist_ins.keys():
        sc = []
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = [np.inner(x, y)/norm(y) for x, y in zip(a, v)] # a along direction of v
            sc.extend(speed_ctrl)
        conditions[dist_ins[dist], angle_ins[angle]] = round(np.mean(sc), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar(orientation="horizontal")
cbar.set_label('Speed control ($m/s^2$)')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.03:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 5), list(angle_ins.keys()))
plt.yticks(range(0, 2), list(dist_ins.keys()))
plt.xlabel("Obstacle path angle (°)")
plt.ylabel("Init obstacle distance (m)")

In [ ]:
'''Heading control by condition (abs angle)'''
%matplotlib qt
load_data('Cohen_movObst2_data')
conditions = np.zeros((2, 5)) # speed by angle
angle_ins = {100: 0, 110: 1, 120: 2, 130: 3, 140: 4}
dist_ins = {4.7: 0, 5.8: 1}
for angle in angle_ins.keys():
    for dist in dist_ins.keys():
        hc = []
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            v = data.info['v_subj'][i][t0:t1]
            a = data.info['a_subj'][i][t0:t1]
            speed_ctrl = np.abs([np.inner(x, y)/norm(y) for x, y in zip(a, v)]) # a along direction of v
            heading_ctrl = np.sqrt(norm(a, axis=1) ** 2 - speed_ctrl ** 2) # a perpendicular to v
            hc.extend(heading_ctrl)
        conditions[dist_ins[dist], angle_ins[angle]] = round(np.mean(hc), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar(orientation="horizontal")
cbar.set_label('Heading control ($m/s^2$)')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] >= 0.25:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 5), list(angle_ins.keys()))
plt.yticks(range(0, 2), list(dist_ins.keys()))
plt.xlabel("Obstacle path angle (°)")
plt.ylabel("Init obstacle distance (m)")

### Cohen_movObst2: Passing order and passing distance

In [ ]:
# Make a pandas df with subj_id, trial_id, obst_angle, obst_speed, pass_order
rows = []

import pandas as pd

for i in range(len(data.trajs)):
    if i not in data.dump:
        subj_id = data.info['subj_id'][i]
        trial_id = data.info['trial_id'][i]
        obst_angle = data.info['obst_angle'][i]
        obst_dist = data.info['obst_dist'][i]
        pass_order = data.info['pass_order'][i]
        rows.append({'subj_id': subj_id, 'trial_id': trial_id, 'obst_angle': obst_angle, 'obst_dist': obst_dist, 'pass_order': pass_order})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(RESULTS_DIR, 'Cohen_movObst2_pass_order.csv'), index=False)

In [ ]:
'''Heatmap and line plot of front passing ratio'''
%matplotlib qt
load_data('Cohen_movObst2_data')
name = data.info['experiment_name']
conditions = np.zeros((2, 5)) # dist by angle
angle_ins = {100: 0, 110: 1, 120: 2, 130: 3, 140: 4}
dist_ins = {4.7: 0, 5.8: 1}
for angle in angle_ins.keys():
    for dist in dist_ins.keys():
        orders = []
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            orders.append(data.info['pass_order'][i])
        conditions[dist_ins[dist], angle_ins[angle]] = round(np.mean(np.array(orders) > 0), 2)

# Heatmap
fig, ax = plt.subplots(1, 1, constrained_layout=True)
plt.imshow(conditions, cmap='plasma')
cbar = plt.colorbar(orientation="horizontal")
cbar.set_label('Ahead passing ratio')
for i in range(conditions.shape[0]):
    for j in range(conditions.shape[1]):
        if conditions[i, j] > 0.7:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="k")
        else:
            text = plt.text(j, i, str(conditions[i, j]),
                           ha="center", va="center", color="w")
plt.xticks(range(0, 5), list(angle_ins.keys()))
plt.yticks(range(0, 2), list(dist_ins.keys()))
plt.xlabel("Obstacle path angle (°)")
plt.ylabel("Init obstacle dist (m)")


In [ ]:
'''Minimum Predicted Distance'''
%matplotlib qt
load_data('Cohen_movObst2_data')
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5), constrained_layout=True)
ax.set_title('Data')
ax.set_ylabel('SMPD (m)')
ax.set_xlabel('normalized time (%)')
ax.set_ylim((-3, 3))
last_smpd_front = []
last_smpd_behind = []
for i in range(len(data.trajs)):
    if data.info['obst_angle'][i] and data.info['obst_dist'][i] and i not in data.dump: 
        t0 = data.info['obst_onset'][i]
        t1 = data.info['obst_out'][i] - 1
        p0 = data.info['p_subj'][i][t0:t1]
        p1 = data.info['p_obst'][i][t0:t1]
        v0 = data.info['v_subj'][i][t0:t1]
        v1 = data.info['v_obst'][i][t0:t1]
        t = np.linspace(0, 100, len(p0))
        smpd = []
        
        for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
            smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
        x0 = smpd[0]
        for x1 in smpd:
            if abs(x1 - x0) > 1:
                smpd[30:] = np.sign(smpd[30]) * np.abs(smpd[30:])
                break
            x0 = x1
        ax.plot(t, smpd, 'k', linewidth=0.1, alpha=0.5)
        if smpd[-1] > 0:
            last_smpd_front.append(smpd[-1])
        else:
            last_smpd_behind.append(smpd[-1])
print(np.mean(last_smpd_front), np.std(last_smpd_front), np.mean(last_smpd_behind), np.std(last_smpd_behind))
    

In [ ]:
'''Minimum Passing Distance by condition'''
load_data('Cohen_movObst2_data')
angles = sorted(set([abs(x) for x in data.info['obst_angle'] if x]))
dists = sorted(set([abs(x) for x in data.info['obst_dist'] if x]))
fig, axs = plt.subplots(5, 2, figsize=(8, 10), constrained_layout=True)
fig.suptitle('Signed minimum predicted distance (SMPD)')
for ix, angle in enumerate(angles):
    for iy, dist in enumerate(dists):
        axs[ix, iy].set_title(f'angle {angle}°, distance {dist} m')
        axs[ix, iy].set_ylabel('SMPD (m)')
        axs[ix, iy].set_xlabel('normalized time (%)')
        axs[ix, iy].set_ylim((-3, 3))
        for i in range(len(data.trajs)):
            if not data.info['obst_angle'][i]:
                continue
            if angle != abs(data.info['obst_angle'][i]) or dist != data.info['obst_dist'][i] or i in data.dump:
                continue
            t0 = data.info['obst_onset'][i]
            t1 = data.info['obst_out'][i]
            p0 = data.info['p_subj'][i][t0:t1]
            p1 = data.info['p_obst'][i][t0:t1]
            v0 = data.info['v_subj'][i][t0:t1]
            v1 = data.info['v_obst'][i][t0:t1]
            t = np.linspace(0, 100, len(p0))
            smpd = []
            for _p0, _p1, _v0, _v1 in zip(p0, p1, v0, v1):        
                smpd.append(min_sep(_p0, _p1, _v0, _v1)[0])
            # Fix the sign flip in data
            x0 = smpd[0]
            for x1 in smpd:
                if abs(x1 - x0) > 1:
                    smpd[30:] = np.sign(smpd[30]) * np.abs(smpd[30:])
                    break
                x0 = x1
            axs[ix, iy].plot(t, smpd, 'k', linewidth=0.1, alpha=0.7)